# Wan text-to-video server for `Youtube_video_automation`

Backing endpoint for `tools/colab_video.py` / `scripts/produce_reel_clip.py`, and the main
pipeline's AI_VIDEO beats when `AI_VIDEO_PROVIDER=colab` (or `auto`) is set in `.env`.

## Why this notebook does not just call `WanPipeline.from_pretrained(...)`

The obvious version of this notebook **crashes the Colab kernel**, with no Python traceback — the
Jupyter log just shows `restarting kernel (1/5)`. That is the OS `SIGKILL`ing the process for
running out of **system RAM**, and the cause is one component:

| | size |
|---|---|
| UMT5-XXL text encoder (`text_encoder/`) | **22.7 GB** fp32 on disk → **~11.4 GB** in fp16 |
| Wan2.1 transformer (1.3B) | ~2.6 GB fp16 |
| VAE (kept fp32 — Wan's VAE is unstable in half precision) | ~0.25 GB |
| **Free Colab system RAM** | **~12.7 GB** |

`enable_model_cpu_offload()` keeps the text encoder resident *in system RAM* so it can be streamed
to the GPU on demand. On a paid box with 50 GB of RAM that is fine. On free Colab it is ~11.4 GB of
an ~12.7 GB budget, and the kernel dies partway through loading.

The irony is that a free T4 has **more VRAM (16 GB) than it has system RAM (12.7 GB)**. So this
notebook inverts the usual advice and never uses CPU offload at all:

1. **Encode phase** — load *only* UMT5-XXL, straight onto the GPU (`device_map`, so weights stream
   shard-by-shard and CPU RAM never holds more than one ~5 GB shard). Encode the prompt to a pair
   of small embedding tensors.
2. **Free it** — `del` + `empty_cache()`, returning all ~11.4 GB of VRAM.
3. **Denoise phase** — the pipeline is loaded with `text_encoder=None, tokenizer=None` (diffusers
   skips any component explicitly passed as `None`), so only the transformer + VAE are resident,
   and they fit the GPU outright with no offloading.

`WanPipeline.__call__` contains no references to `self.text_encoder` or `self.tokenizer`, so once
`prompt_embeds` is supplied it runs perfectly well without them.

The tradeoff: the encoder is re-read from local disk once per clip (~1 min), since it is not kept
between calls. Repeat prompts are served from `_EMBED_CACHE`. On a Colab Pro high-RAM runtime you
could keep it loaded instead, but the reload is small next to a ~10 min denoise.

## The first run downloads 22.7 GB and that is the slow part

There is no fp16 copy of the text encoder in the repo, so a cold session fetches the full fp32
weights. Two things make that bearable, both handled above:

* `HF_XET_HIGH_PERFORMANCE=1`, because Xet's local *reconstruct* step is CPU-bound and crawls on
  Colab's ~2 vCPU (~11 MB/s observed, while raw transfer was ~94 MB/s).
* Caching `HF_HOME` on Drive, so it is a one-time cost rather than a per-session one.

## Model, chosen from the GPU it lands on

| Runtime | VRAM | Model | Output | Speed |
|---|---|---|---|---|
| Free **T4** | 16 GB | `Wan-AI/Wan2.1-T2V-1.3B-Diffusers` | 832x480 @ 16fps | ~6-12 min / 5 s clip |
| Pro **L4 / A100** | 24-40 GB | `Wan-AI/Wan2.2-TI2V-5B-Diffusers` | 1280x704 @ 24fps | ~5-9 min / 5 s clip |

Both are Apache-2.0 (unrestricted commercial use, unlike LTX's revenue-capped community license).

**dtype** is also picked from the GPU: `bfloat16` has no hardware support before Ampere (`sm_80`),
so on a T4 (`sm_75`) torch accepts it and silently runs an emulated slow path. Pre-Ampere gets
`float16` instead.

## How to use
1. `Runtime -> Change runtime type -> T4 GPU` (or L4/A100 on Colab Pro), then `Run all`.
2. Wait for the last cell to print `https://xxxxxxxx.gradio.live` (first run downloads 10-25 GB).
3. Put that URL in the project's `.env` as `COLAB_VIDEO_URL=...` -- **it changes every restart**.
4. Keep the tab open. Free sessions drop after ~90 min idle and ~12 h hard cap; when the session
   dies the URL 404s and the pipeline falls AI_VIDEO beats back to the mascot rather than failing.


In [ ]:
# Wan2.1 needs diffusers >= 0.33; Wan2.2-TI2V-5B needs >= 0.35. accelerate is required for the
# device_map streaming load in the encode cell. sentencepiece + protobuf are for the UMT5
# tokenizer; ftfy for prompt cleanup; imageio-ffmpeg to write the mp4.
!pip install -q -U "diffusers>=0.35.0" transformers accelerate ftfy sentencepiece protobuf \
    imageio imageio-ffmpeg gradio gradio_client


## Optional but strongly recommended: persist the model cache to Drive

The text encoder alone is **22.7 GB** and Colab wipes its disk between sessions, so by default you
re-download it *every* session. Mounting Drive and pointing `HF_HOME` at it makes that a one-time
cost — later sessions start in seconds.

It needs ~30 GB of Drive space, which is more than the free 15 GB tier. Skip this cell if you
don't have the room; everything still works, it just re-downloads each time.


In [ ]:
# Optional. Run this BEFORE the next cell if you want the download cached across sessions.
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
import os

# Set before anything imports huggingface_hub, or they are ignored.
#
# The UMT5-XXL text encoder is a 22.7 GB fp32 download (there is no fp16 copy in the repo) and HF
# now serves it over Xet, which fetches many small byte-ranges and *reconstructs* the file locally.
# That reconstruct step is CPU-bound, and free Colab has ~2 vCPU -- it runs at ~11 MB/s even while
# raw transfer sits at ~94 MB/s, turning a few minutes into a couple of hours. High-performance
# mode parallelises it. (HF_HUB_ENABLE_HF_TRANSFER is ignored by current huggingface_hub; and
# HF_HUB_DISABLE_XET=1 falls back to plain HTTP, which is a last resort -- slower still.)
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

# Point the HF cache at Drive (see the optional cell above) so the 22.7 GB survives a restart.
if os.path.isdir("/content/drive/MyDrive"):
    os.environ.setdefault("HF_HOME", "/content/drive/MyDrive/hf_cache")
    print(f"HF cache -> {os.environ['HF_HOME']} (persists across sessions)")
else:
    print("HF cache -> ephemeral session disk; the 22.7 GB encoder re-downloads next session.")

import torch

CAPABILITY = (0, 0)
VRAM_GB = 0.0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

try:
    import psutil

    RAM_GB = psutil.virtual_memory().total / 1e9
except Exception:
    RAM_GB = 0.0

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    CAPABILITY = (props.major, props.minor)
    VRAM_GB = props.total_memory / 1e9
    print(f"GPU: {props.name}  ({VRAM_GB:.0f} GB VRAM, compute capability {props.major}.{props.minor})")
    print(f"System RAM: {RAM_GB:.0f} GB")
    if CAPABILITY < (8, 0):
        print("  pre-Ampere: no bfloat16 hardware -> using float16 instead.")
    if VRAM_GB < 20:
        print("  <20 GB VRAM -> Wan2.1-T2V-1.3B (832x480). Switch to an L4/A100 runtime for 720p.")
    else:
        print("  >=20 GB VRAM -> Wan2.2-TI2V-5B (1280x704 @ 24fps).")
    if 0 < RAM_GB < 20:
        print(f"  {RAM_GB:.0f} GB RAM is less than the ~11.4 GB fp16 text encoder needs to sit in")
        print("  comfortably -- hence the two-phase encode below. Do NOT add enable_model_cpu_offload().")
else:
    print("NO GPU. Wan on CPU is impractically slow (hours/clip). "
          "Runtime > Change runtime type > T4 GPU.")


In [ ]:
import time

import torch
from diffusers import AutoencoderKLWan, UniPCMultistepScheduler, WanPipeline

# Chosen from the probe cell above rather than hardcoded -- see the notebook header.
BIG_GPU = VRAM_GB >= 20
MODEL_ID = "Wan-AI/Wan2.2-TI2V-5B-Diffusers" if BIG_GPU else "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
DTYPE = torch.bfloat16 if CAPABILITY >= (8, 0) else torch.float16
FLOW_SHIFT = 5.0 if BIG_GPU else 3.0  # Wan guidance: 3.0 for <=480p, 5.0 for 720p
MAX_SEQUENCE_LENGTH = 512             # WanPipeline.__call__'s own default; the encode cell must match

# Native output geometry for the loaded model. tools/colab_video.py sends its own height/width
# from settings; these are the Gradio defaults and what `generate` rounds toward.
DEFAULT_W, DEFAULT_H = (1280, 704) if BIG_GPU else (832, 480)
DEFAULT_FRAMES = 121 if BIG_GPU else 81   # ~5 s at 24fps / 16fps respectively
EXPORT_FPS = 24 if BIG_GPU else 16        # each model's training fps

print(f"Loading {MODEL_ID} (transformer + VAE only) in {DTYPE}...", flush=True)
_t0 = time.time()

# The VAE is loaded in fp32 on purpose (Wan's VAE is numerically unstable in fp16/bf16).
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)

# text_encoder/tokenizer are explicitly None: diffusers' from_pretrained skips loading any
# component passed as None, so the 22.7 GB text encoder is never downloaded into this pipeline at
# all. The encode cell below loads it separately, on the GPU, and frees it again.
pipe = WanPipeline.from_pretrained(
    MODEL_ID,
    vae=vae,
    text_encoder=None,
    tokenizer=None,
    torch_dtype=DTYPE,
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=FLOW_SHIFT)

# No enable_model_cpu_offload() on purpose. Without the text encoder the resident set is just the
# transformer + VAE, which fits the GPU outright -- and offloading would put it back in the scarce
# resource (system RAM) while also slowing every denoise step down with host<->device streaming.
pipe.to(DEVICE)

try:
    pipe.vae.enable_tiling()
except AttributeError:
    pass

print(f"Loaded in {time.time() - _t0:.0f}s.  Output: {DEFAULT_W}x{DEFAULT_H} @ {EXPORT_FPS}fps", flush=True)
if torch.cuda.is_available():
    print(f"VRAM in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB", flush=True)


In [ ]:
import time

from huggingface_hub import snapshot_download

# Pull the text encoder + tokenizer up front, as their own visible step.
#
# _encode() below would fetch these on its first call anyway, but burying a 22.7 GB download inside
# the first generation makes a working notebook look hung -- and on a fresh session it is by far the
# longest single operation here. Doing it explicitly also means a failed/throttled download surfaces
# now rather than as a mysteriously slow first clip.
#
# This is a no-op once cached (instant on a Drive-backed HF_HOME).
print("Prefetching text encoder + tokenizer (22.7 GB on a cold cache)...", flush=True)
_t0 = time.time()
snapshot_download(MODEL_ID, allow_patterns=["text_encoder/*", "tokenizer/*"])
print(f"Text encoder ready in {time.time() - _t0:.0f}s.", flush=True)


In [ ]:
import gc
import tempfile
import time

from diffusers.utils import export_to_video
from transformers import AutoTokenizer, UMT5EncoderModel

NUM_INFERENCE_STEPS = 25  # Wan's default is 50; 25 roughly halves runtime at a modest quality cost

DEFAULT_NEGATIVE = (
    "overexposed, static, blurred details, subtitles, worst quality, low quality, JPEG artifacts, "
    "ugly, deformed, disfigured, misshapen limbs, fused fingers, still picture, cluttered background, "
    "watermark, text, logo"
)

# Embeddings are tiny (a couple of MB) and the encoder reload is the expensive part, so a repeat
# prompt -- a retried beat, a re-run of the same story -- skips the reload entirely.
_EMBED_CACHE: dict = {}


def _log(msg: str) -> None:
    print(f"[colab_video {time.strftime('%H:%M:%S')}] {msg}", flush=True)


def _round_frames(n: int) -> int:
    """Wan's VAE has temporal stride 4 -> num_frames must be 4k+1."""
    n = max(5, int(n))
    return ((n - 1) // 4) * 4 + 1


def _encode(prompt: str, negative_prompt: str):
    """Load UMT5-XXL onto the GPU, encode, and free it again.

    device_map streams the shards straight to VRAM, so host RAM never holds more than one ~5 GB
    shard -- loading it to CPU instead (which is what enable_model_cpu_offload does) needs ~11.4 GB
    resident and is what SIGKILLs the kernel on free Colab. See the notebook header.
    """
    key = (prompt, negative_prompt)
    if key in _EMBED_CACHE:
        _log("prompt embeddings served from cache")
        return _EMBED_CACHE[key]

    t0 = time.time()
    _log("loading text encoder onto the GPU...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
    text_encoder = UMT5EncoderModel.from_pretrained(
        MODEL_ID, subfolder="text_encoder", torch_dtype=DTYPE, device_map={"": DEVICE}
    )
    # An encode-only view of the pipeline: transformer/transformer_2 are optional components and
    # vae is passed as None, so nothing else is loaded or moved.
    encoder_pipe = WanPipeline.from_pretrained(
        MODEL_ID,
        tokenizer=tokenizer,
        text_encoder=text_encoder,
        vae=None,
        transformer=None,
        torch_dtype=DTYPE,
    )
    try:
        prompt_embeds, negative_embeds = encoder_pipe.encode_prompt(
            prompt=prompt,
            negative_prompt=negative_prompt,
            do_classifier_free_guidance=True,
            max_sequence_length=MAX_SEQUENCE_LENGTH,
            device=DEVICE,
            dtype=DTYPE,
        )
    finally:
        del encoder_pipe, text_encoder, tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    _log(f"encoded + freed encoder in {time.time() - t0:.0f}s")
    _EMBED_CACHE[key] = (prompt_embeds, negative_embeds)
    return prompt_embeds, negative_embeds


def generate(prompt: str, negative_prompt: str, num_frames: int, height: int, width: int, guidance_scale: float) -> str:
    num_frames = _round_frames(num_frames)
    height = max(256, int(round(height / 16) * 16))
    width = max(256, int(round(width / 16) * 16))
    neg = (negative_prompt or "").strip() or DEFAULT_NEGATIVE

    _log(f"generate: {num_frames} frames  {width}x{height}  steps={NUM_INFERENCE_STEPS}  gs={guidance_scale}")
    _log(f"prompt: {prompt[:200]}")
    t0 = time.time()
    try:
        prompt_embeds, negative_embeds = _encode(prompt, neg)
        frames = pipe(
            prompt_embeds=prompt_embeds,
            negative_prompt_embeds=negative_embeds,
            num_frames=num_frames,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=float(guidance_scale),
            height=height,
            width=width,
        ).frames[0]
        _log(f"denoise done in {time.time() - t0:.0f}s -- exporting {len(frames)} frames @ {EXPORT_FPS}fps")
        out_path = tempfile.mktemp(suffix=".mp4")
        export_to_video(frames, out_path, fps=EXPORT_FPS)
        _log(f"wrote {out_path}  (total {time.time() - t0:.0f}s)")
        return out_path
    except Exception as e:
        _log(f"FAILED after {time.time() - t0:.0f}s -- {type(e).__name__}: {e}")
        raise
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
# Optional but recommended on a fresh session: one short clip end to end, so an OOM or a bad
# install surfaces here rather than as a dead endpoint the pipeline silently falls back from.
# 25 frames is ~1.5 s -- a couple of minutes rather than ten.
_smoke = generate(
    "a tiny glowing firefly drifting through a dark forest at night, soft golden light, "
    "children's storybook animation style",
    DEFAULT_NEGATIVE,
    25,
    DEFAULT_H,
    DEFAULT_W,
    5.0,
)
print("smoke test OK ->", _smoke)


In [ ]:
import gradio as gr

# Input order MUST match tools/colab_video.py's positional client.predict(...) call exactly:
# prompt, negative_prompt, num_frames, height, width, guidance_scale.
#
# api_name is pinned explicitly: gr.Interface defaults it to "predict" on Gradio <=5.x but to the
# wrapped function's name on 6.x, and the install cell doesn't pin a gradio version. Naming it here
# makes the endpoint "/generate" on every version. (tools/colab_video.py probes both names anyway,
# so a session started from an older copy of this notebook keeps working.)
demo = gr.Interface(
    fn=generate,
    inputs=[
        gr.Textbox(label="prompt", lines=3),
        gr.Textbox(label="negative_prompt", value=DEFAULT_NEGATIVE),
        gr.Number(label="num_frames", value=DEFAULT_FRAMES, precision=0),
        gr.Number(label="height", value=DEFAULT_H, precision=0),
        gr.Number(label="width", value=DEFAULT_W, precision=0),
        gr.Number(label="guidance_scale", value=5.0),
    ],
    outputs=gr.Video(label="output"),
    title=MODEL_ID.split("/")[-1],
    description="Offline AI_VIDEO endpoint for Youtube_video_automation (set COLAB_VIDEO_URL in .env)",
    api_name="generate",
)

demo.launch(share=True, show_error=True)
# Copy the printed https://....gradio.live URL (NOT the localhost one) into .env as COLAB_VIDEO_URL.
